<a href="https://colab.research.google.com/github/aanyam2007/-git-lab-demo/blob/main/MNISTDigitalClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
import matplotlib.pyplot as plt
import torch.optim as optim
from torchvision.transforms import ToTensor

In [ ]:
train_data=datasets.MNIST(root="data",train=True,download=True,transform=ToTensor())
test_data=datasets.MNIST(root="data",train=False,download=True,transform=ToTensor())

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.7MB/s]


In [ ]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [ ]:
train_data.data.shape

torch.Size([60000, 28, 28])

In [ ]:
test_data.data.shape

torch.Size([10000, 28, 28])

In [ ]:
#feeds data to model in batches
loader={
    'train': DataLoader(train_data,batch_size=64,shuffle=True),
    'test': DataLoader(test_data,batch_size=64,shuffle=False)
}

In [ ]:
loader

{'train': <torch.utils.data.dataloader.DataLoader at 0x7fc3839941d0>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7fc4b7b54950>}

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        # Convolution layers
        self.conv1=nn.Conv2d(1,10,kernel_size=5)
        self.conv2=nn.Conv2d(10,20,kernel_size=5)
        #Dropout for regularization
        self.conv2_drop=nn.Dropout2d()

        #Fully connected layers
        self.fc1=nn.Linear(320,50)
        self.fc2=nn.Linear(50,10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        # ✅ FLATTEN HERE
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


In [ ]:
model=CNN()
optimizer=optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()

In [ ]:
epochs=5
for epoch in range(epochs):
    model.train()
    for images,labels in loader['train']:
        optimizer.zero_grad()
        #forward pass
        output=model(images)
        #compute loss
        loss=loss_fn(output,labels)
        #backward pass
        loss.backward()
        #new weights
        optimizer.step()

In [ ]:
model.eval() #turns off dropout
correct = 0
total = 0

with torch.no_grad():
    for images, labels in loader['test']:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy}%")


Accuracy: 98.69%
